# Variety LLM Integration Tests
6 end-to-end tests. Run from `CAPSTONE2026_SPRING/` root with Ollama running.

In [1]:
import importlib
import intake_engine.policies.complaint_policies
importlib.reload(intake_engine.policies.complaint_policies)
import intake_engine.policies.target_specs, intake_engine.policies.base_complaint_policy
import intake_engine.policies.policy_selector, intake_engine.state.templates
import intake_engine.state.routing, intake_engine.state.rule_based_parser
import intake_engine.llm_extractor, intake_engine.llm_adapter
import intake_engine.IntakeState, intake_engine.app_flow
for mod in [
    intake_engine.policies.target_specs, intake_engine.policies.base_complaint_policy,
    intake_engine.policies.complaint_policies, intake_engine.policies.policy_selector,
    intake_engine.state.templates, intake_engine.state.routing,
    intake_engine.state.rule_based_parser, intake_engine.llm_extractor,
    intake_engine.llm_adapter, intake_engine.IntakeState, intake_engine.app_flow,
]:
    importlib.reload(mod)
print('Reloaded.')

Reloaded.


In [2]:
from pprint import pprint
from intake_engine.app_flow import IntakeAppFlow
from intake_engine.policies.complaint_policies import COMPLAINT_POLICIES
print(f'Policies loaded: {len(COMPLAINT_POLICIES)}')

all_results = []

def run_test(session_name, complaint, answers, label=None, checks=None):
    label = label or session_name
    print('=' * 70)
    print(f'TEST: {label}')
    print('=' * 70)
    app = IntakeAppFlow(conn=None)
    app.new_session(session_name=session_name, auto_save=False)
    app.state.data['chief_complaint_primary'] = complaint
    app.start_intake(auto_save=False)
    intents = []
    for answer in answers:
        result = app.submit_answer(answer, auto_save=False)
        intents.append(result['next_intent'])
    state = app.get_state()
    print('\nINTENT SEQUENCE:')
    for i, intent in enumerate(intents):
        print(f'  {i+1:>2}. {intent}')
    print('\nHPI:')
    pprint({k: v for k, v in state['hpi'].items() if v and v != []})
    print('\nPOLICY ANSWERS (non-None):')
    pprint({k: v for k, v in state['policy_answers'].items() if v is not None})
    print('\nFLAGS:', state['flags'])
    print('MISSING:', state['missing_clarifications'])
    print('INTAKE COMPLETE:', state['conversation_meta']['intake_complete'])
    passed = []
    if checks:
        print('\nCHECKS:')
        for label_c, condition in checks(state, intents):
            status = '\033[92m PASS\033[0m' if condition else '\033[91m FAIL\033[0m'
            print(f'  {status}  {label_c}')
            passed.append(condition)
    print()
    return app, state, intents, passed

print('Helper loaded.')

Policies loaded: 19
Helper loaded.


## 1. Seizure — known epileptic, missed medication

In [3]:
def seizure_checks(state, intents):
    pa = state['policy_answers']
    return [
        ('Policy routes to seizure', 'ask_head_trauma' in intents),
        ('seizure_history captured as True', pa.get('seizure_history') is True),
        ('antiepileptic_compliance captured as False', pa.get('antiepileptic_compliance') is False),
        ('tongue_or_lip_biting captured', pa.get('tongue_or_lip_biting') is not None),
        ('postictal_confusion captured', pa.get('postictal_confusion') is not None),
        ('No consecutive loops', all(intents[i] != intents[i-1] for i in range(1, len(intents)))),
    ]

app, state, intents, passed = run_test(
    session_name='seizure_variety',
    complaint='i had a seizure',
    label='SEIZURE — known epileptic, missed medication',
    answers=[
        'No weakness or numbness now.',               # neurologic_symptoms
        'No head injury.',                            # head_trauma
        'No fever or stiff neck.',                    # fever_or_neck_stiffness
        'Yes, confused for about 15 minutes after.',  # confusion_or_ams
        'No, just the one episode.',                  # syncope_or_presyncope
        'No, it has not happened again.',             # rapid_worsening
        'My wife witnessed it, no warning beforehand.', # prodrome_witnessed
        'About two hours ago.',                       # onset
        'The shaking lasted about 90 seconds.',       # duration
        '8 out of 10.',                               # severity -> safety check
        'No, I am okay.',                             # ask_immediate_safety_check
        'Just the one episode.',                      # timing
        'Resolved now but I feel very tired.',        # course
        'Nothing triggered it that I know of.',       # aggravating_factors
        'Rest is helping.',                           # relieving_factors
        'Muscle soreness and fatigue.',               # associated_symptoms
        'Levetiracetam 500mg twice daily.',           # medications
        'No allergies.',                              # allergies
        'Yes, I have epilepsy.',                      # seizure_history
        'No, I ran out of levetiracetam three days ago.', # antiepileptic_compliance
        'No, I slept normally.',                      # recent_sleep_deprivation
        'No alcohol or drugs.',                       # recent_substance_use
        'Yes, I bit my tongue.',                      # tongue_or_lip_biting
        'Yes, I lost bladder control.',               # incontinence_during_event
        'Yes, confused for about 15 minutes.',        # postictal_confusion
    ],
    checks=seizure_checks,
)
all_results.extend(passed)

TEST: SEIZURE — known epileptic, missed medication

INTENT SEQUENCE:
   1. ask_head_trauma
   2. ask_fever_or_neck_stiffness
   3. ask_confusion_or_ams
   4. ask_syncope_or_presyncope
   5. ask_rapid_worsening
   6. ask_prodrome_witnessed_loss_of_consciousness
   7. ask_onset
   8. ask_duration
   9. ask_severity
  10. ask_immediate_safety_check
  11. ask_timing
  12. ask_course
  13. ask_aggravating_factors
  14. ask_relieving_factors
  15. ask_associated_symptoms
  16. ask_medications
  17. ask_allergies
  18. ask_seizure_history
  19. ask_antiepileptic_compliance
  20. ask_recent_sleep_deprivation
  21. ask_recent_substance_use
  22. ask_tongue_or_lip_biting
  23. ask_incontinence_during_event
  24. ask_postictal_confusion
  25. summarize_and_check_for_anything_else

HPI:
{'associated_symptoms': ['muscle soreness', 'fatigue'],
 'course': 'Resolved now but I feel very tired',
 'duration': 'the shaking lasted about 90 seconds',
 'onset': 'About two hours ago',
 'relieving_factors': ['

## 2. Fever — meningitis red flag scenario

In [4]:
def fever_checks(state, intents):
    pa = state['policy_answers']
    return [
        ('Policy routes to fever', 'ask_neurologic_symptoms' in intents),
        ('fever_or_neck_stiffness captured as True', pa.get('fever_or_neck_stiffness') is True),
        ('rash_or_petechiae asked', 'ask_rash_or_petechiae' in intents),
        ('HPI onset captured', bool(state['hpi'].get('onset'))),
        ('No consecutive loops', all(intents[i] != intents[i-1] for i in range(1, len(intents)))),
    ]

app, state, intents, passed = run_test(
    session_name='fever_variety',
    complaint='fever',
    label='FEVER — meningitis red flag scenario',
    answers=[
        'Yes, fever and neck stiffness.',            # fever_or_neck_stiffness
        'I feel a bit confused.',                    # neurologic_symptoms
        'No shortness of breath.',                   # shortness_of_breath
        'No chest pain.',                            # chest_pain
        'Yes, I am confused and not myself.',        # confusion_or_ams
        'Yes, getting worse over the last hour.',    # rapid_worsening
        'Yesterday evening.',                        # onset
        'About 18 hours.',                           # duration
        '8 out of 10.',                              # severity -> safety check
        'No, I am okay.',                            # ask_immediate_safety_check
        'Constant and worsening.',                   # timing
        'Getting worse.',                            # course
        'Nothing helps.',                            # aggravating_factors
        'Nothing is helping.',                       # relieving_factors
        'Severe headache and sensitivity to light.', # associated_symptoms
        'No medications.',                           # medications
        'No allergies.',                             # allergies
        'No sick contacts.',                         # sick_contacts
        'No cough.',                                 # cough
        'No urinary symptoms.',                      # urinary_symptoms
        'Yes, I noticed a rash with small spots.',   # rash_or_petechiae
    ],
    checks=fever_checks,
)
all_results.extend(passed)

TEST: FEVER — meningitis red flag scenario

INTENT SEQUENCE:
   1. ask_neurologic_symptoms
   2. ask_shortness_of_breath
   3. ask_chest_pain
   4. ask_confusion_or_ams
   5. ask_rapid_worsening
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_immediate_safety_check
  10. ask_timing
  11. ask_aggravating_factors
  12. ask_relieving_factors
  13. ask_associated_symptoms
  14. ask_medications
  15. ask_allergies
  16. ask_nausea_or_vomiting
  17. ask_cough
  18. ask_urinary_symptoms
  19. ask_rash_or_petechiae
  20. ask_sick_contacts
  21. summarize_and_check_for_anything_else

HPI:
{'aggravating_factors': ['getting worse'],
 'duration': 'about 18 hours',
 'onset': 'Yesterday evening',
 'severity': '8/10',
 'timing': 'Constant and worsening'}

POLICY ANSWERS (non-None):
{'aura_features': [],
 'chest_pain': False,
 'confusion_or_ams': True,
 'cough': False,
 'fever_or_neck_stiffness': True,
 'nausea_or_vomiting': False,
 'neurologic_symptom_terms': ['confusion'],
 'neurolog

## 3. Leg pain / swelling — DVT scenario

In [5]:
def leg_checks(state, intents):
    pa = state['policy_answers']
    return [
        ('calf_tenderness_or_warmth asked', 'ask_calf_tenderness_or_warmth' in intents),
        ('calf_tenderness_or_warmth captured as True', pa.get('calf_tenderness_or_warmth') is True),
        ('unilateral_leg_swelling captured as True', pa.get('unilateral_leg_swelling') is True),
        ('recent_immobility_or_travel asked', 'ask_recent_immobility_or_travel' in intents),
        ('HPI location captured', bool(state['hpi'].get('location'))),
        ('No consecutive loops', all(intents[i] != intents[i-1] for i in range(1, len(intents)))),
    ]

app, state, intents, passed = run_test(
    session_name='leg_variety',
    complaint='my left leg is swollen',
    label='LEG PAIN / SWELLING — DVT scenario after long flight',
    answers=[
        'No, not getting worse quickly.',            # rapid_worsening
        'Yes, tender and warm to touch.',            # calf_tenderness_or_warmth
        'Yes, just the left leg.',                   # unilateral_leg_swelling
        'No shortness of breath.',                   # shortness_of_breath
        'No chest pain.',                            # chest_pain
        'No fainting.',                              # syncope_or_presyncope
        'No weakness or numbness.',                  # neurologic_symptoms
        'Yesterday morning.',                        # onset
        'About 24 hours.',                           # duration
        '6 out of 10.',                              # severity (no safety check at 6/10)
        'Left calf.',                                # location
        'Throbbing and tight.',                      # character
        'Walking makes it worse.',                   # aggravating_factors
        'Elevation helps slightly.',                 # relieving_factors
        'No other symptoms.',                        # associated_symptoms
        'Aspirin daily.',                            # medications
        'No allergies.',                             # allergies
        'Just the left leg.',                        # unilateral_vs_bilateral
        'Yes, I was on a 10 hour flight two days ago.', # recent_immobility_or_travel
        'No recent surgery or injury.',              # recent_trauma_or_surgery
        'No heavy lifting.',                         # recent_heavy_lifting
    ],
    checks=leg_checks,
)
all_results.extend(passed)

TEST: LEG PAIN / SWELLING — DVT scenario after long flight

INTENT SEQUENCE:
   1. ask_calf_tenderness_or_warmth
   2. ask_unilateral_leg_swelling
   3. ask_shortness_of_breath
   4. ask_chest_pain
   5. ask_syncope_or_presyncope
   6. ask_neurologic_symptoms
   7. ask_onset
   8. ask_duration
   9. ask_severity
  10. ask_location
  11. ask_character
  12. ask_aggravating_factors
  13. ask_relieving_factors
  14. ask_associated_symptoms
  15. ask_medications
  16. ask_allergies
  17. ask_unilateral_vs_bilateral
  18. ask_recent_immobility_or_travel
  19. ask_recent_trauma_or_surgery
  20. ask_recent_heavy_lifting
  21. summarize_and_check_for_anything_else

HPI:
{'aggravating_factors': ['walking makes it worse'],
 'character': 'Throbbing and tight',
 'duration': 'about 24 hours',
 'location': 'Left calf',
 'onset': 'Yesterday morning',
 'relieving_factors': ['elevation helps slightly'],
 'severity': '6/10'}

POLICY ANSWERS (non-None):
{'aura_features': [],
 'calf_tenderness_or_warmth':

## 4. Eye pain — acute angle closure glaucoma scenario

In [6]:
def eye_checks(state, intents):
    pa = state['policy_answers']
    return [
        # was: ('visual_changes asked', 'ask_visual_changes' in intents),
        ('visual_changes resolved', pa.get('visual_changes') is True),  # ← captured, not necessarily asked
        ('floaters_or_flashes asked', 'ask_floaters_or_flashes' in intents),
        ('photophobia captured as True', pa.get('photophobia') is True),
        ('headache_with_eye_pain asked', 'ask_headache_with_eye_pain' in intents),
        ('HPI severity captured', bool(state['hpi'].get('severity'))),
        ('No consecutive loops', all(intents[i] != intents[i-1] for i in range(1, len(intents)))),
    ]

app, state, intents, passed = run_test(
    session_name='eye_variety',
    complaint='my right eye hurts',
    label='EYE PAIN — acute angle closure glaucoma scenario',
    answers=[
        'Yes, very blurry in my right eye.',         # visual_changes
        'Yes, came on suddenly.',                    # sudden_severe_onset
        'Yes, seeing halos around lights.',          # floaters_or_flashes
        'Vision is dim and partially blocked.',      # vision_loss_pattern
        'No eye injury.',                            # recent_eye_trauma
        'Yes, very sensitive to light.',             # photophobia
        'Two hours ago.',                            # onset
        'About two hours.',                          # duration
        '9 out of 10.',                              # severity -> safety check
        'No, I am okay.',                            # ask_immediate_safety_check
        'Right eye only.',                           # location
        'Sharp pressure pain.',                      # character
        'Light makes it much worse.',                # aggravating_factors
        'Nothing helps.',                            # relieving_factors
        'Nausea and severe headache.',               # associated_symptoms
        'No medications.',                           # medications
        'No allergies.',                             # allergies
        'Pressure type pain.',                       # eye_pain_type
        'No discharge.',                             # eye_discharge
        'Yes, very red.',                            # redness
        'Yes, severe headache.',                     # headache_with_eye_pain
        'No contact lenses.',                        # contact_lens_use
    ],
    checks=eye_checks,
)
all_results.extend(passed)

TEST: EYE PAIN — acute angle closure glaucoma scenario

INTENT SEQUENCE:
   1. ask_sudden_severe_onset
   2. ask_floaters_or_flashes
   3. ask_vision_loss_pattern
   4. ask_recent_eye_trauma
   5. ask_photophobia
   6. ask_onset
   7. ask_duration
   8. ask_severity
   9. ask_immediate_safety_check
  10. ask_location
  11. ask_character
  12. ask_aggravating_factors
  13. ask_relieving_factors
  14. ask_associated_symptoms
  15. ask_medications
  16. ask_allergies
  17. ask_eye_pain_type
  18. ask_eye_discharge
  19. ask_redness
  20. ask_nausea
  21. ask_headache_with_eye_pain
  22. ask_contact_lens_use

HPI:
{'aggravating_factors': ['light makes it much worse'],
 'associated_symptoms': ['nausea', 'severe headache'],
 'character': 'Sharp pressure pain',
 'duration': 'about two hours',
 'location': 'Right eye only',
 'onset': 'Two hours ago',
 'severity': '9/10'}

POLICY ANSWERS (non-None):
{'aura_features': [],
 'eye_discharge': False,
 'eye_pain_type': 'Pressure type pain',
 'floater

## 5. Altered mental status — caregiver-reported, stroke concern

In [7]:
def ams_checks(state, intents):
    pa = state['policy_answers']
    return [
        # was: ('sudden_severe_onset asked', 'ask_sudden_severe_onset' in intents),
        ('sudden_severe_onset resolved', pa.get('sudden_severe_onset') is True),  # ← implicit capture
        ('neurologic_symptoms asked', 'ask_neurologic_symptoms' in intents),
        ('medications NOT in first 7 intents', 'ask_medications' not in intents[:7]),
        ('HPI onset captured', bool(state['hpi'].get('onset'))),
        ('No consecutive loops', all(intents[i] != intents[i-1] for i in range(1, len(intents)))),
    ]

app, state, intents, passed = run_test(
    session_name='ams_variety',
    complaint='my husband is confused and not acting right',
    label='ALTERED MENTAL STATUS — caregiver-reported, stroke concern',
    answers=[
        'Yes, it came on very suddenly.',            # sudden_severe_onset
        'No head injury.',                           # head_trauma
        'No fever or neck stiffness.',               # fever_or_neck_stiffness
        'No breathing trouble.',                     # shortness_of_breath
        'No fainting.',                              # syncope_or_presyncope
        'Yes, getting worse since it started.',      # rapid_worsening
        'Yes, his right arm seems weak.',            # neurologic_symptoms
        'About an hour ago.',                        # onset
        'About one hour.',                           # duration
        '9 out of 10.',                              # severity -> safety check
        'No, he is stable.',                         # ask_immediate_safety_check
        'Getting progressively worse.',              # timing
        'Worsening rapidly.',                        # course
        'Nothing obvious.',                          # aggravating_factors
        'Nothing helps.',                            # relieving_factors
        'Arm weakness and slurred speech.',          # associated_symptoms
        'Metformin and lisinopril.',                 # medications
        'Penicillin.',                               # allergies
        'He last ate breakfast.',                    # last_oral_intake
        'No sick contacts.',                         # sick_contacts
        'No nausea.',                                # nausea_or_vomiting
    ],
    checks=ams_checks,
)
all_results.extend(passed)

TEST: ALTERED MENTAL STATUS — caregiver-reported, stroke concern

INTENT SEQUENCE:
   1. ask_head_trauma
   2. ask_fever_or_neck_stiffness
   3. ask_shortness_of_breath
   4. ask_syncope_or_presyncope
   5. ask_rapid_worsening
   6. ask_neurologic_symptoms
   7. ask_onset
   8. ask_duration
   9. ask_severity
  10. ask_immediate_safety_check
  11. ask_timing
  12. ask_course
  13. ask_aggravating_factors
  14. ask_relieving_factors
  15. ask_associated_symptoms
  16. ask_medications
  17. ask_allergies
  18. ask_last_oral_intake
  19. ask_sick_contacts
  20. ask_nausea_or_vomiting
  21. summarize_and_check_for_anything_else

HPI:
{'associated_symptoms': ['arm weakness', 'slurred speech'],
 'course': 'Worsening rapidly',
 'duration': 'about one hour',
 'onset': 'About an hour ago',
 'severity': '9/10',
 'timing': 'Getting progressively worse'}

POLICY ANSWERS (non-None):
{'aura_features': [],
 'fever_or_neck_stiffness': False,
 'head_trauma': False,
 'last_oral_intake': 'breakfast',
 'n

## 6. Palpitations — exertional, with presyncope

In [8]:
def palp_checks(state, intents):
    pa = state['policy_answers']
    return [
        # was: ('chest_pain asked', 'ask_chest_pain' in intents),
        ('chest_pain resolved', pa.get('chest_pain') is not None),  # ← captured implicitly
        ('syncope_or_presyncope asked', 'ask_syncope_or_presyncope' in intents),
        ('syncope_or_presyncope captured as True', pa.get('syncope_or_presyncope') is True),
        ('diaphoresis asked', 'ask_diaphoresis' in intents),
        ('HPI timing captured', bool(state['hpi'].get('timing'))),
        ('No consecutive loops', all(intents[i] != intents[i-1] for i in range(1, len(intents)))),
    ]

app, state, intents, passed = run_test(
    session_name='palp_variety',
    complaint='heart racing',
    label='PALPITATIONS — exertional, with near-fainting',
    answers=[
        'No chest pain.',                            # chest_pain
        'No trouble breathing.',                     # shortness_of_breath
        'Yes, I nearly fainted during the episode.', # syncope_or_presyncope
        'No, it passed after I stopped exercising.', # rapid_worsening
        'Yes, I was sweating heavily.',              # diaphoresis
        'This afternoon during my run.',             # onset
        'About 5 minutes.',                          # duration
        '7 out of 10.',                              # severity (no safety check at 7/10)
        'Intermittent, comes on with exercise.',     # timing
        'Racing and pounding.',                      # character
        'Exercise brings it on.',                    # aggravating_factors
        'Stopping exercise helps.',                  # relieving_factors
        'Dizziness during the episode.',             # associated_symptoms
        'No medications.',                           # medications
        'No allergies.',                             # allergies
    ],
    checks=palp_checks,
)
all_results.extend(passed)

TEST: PALPITATIONS — exertional, with near-fainting

INTENT SEQUENCE:
   1. ask_shortness_of_breath
   2. ask_syncope_or_presyncope
   3. ask_rapid_worsening
   4. ask_diaphoresis
   5. ask_onset
   6. ask_duration
   7. ask_severity
   8. ask_timing
   9. ask_character
  10. ask_aggravating_factors
  11. ask_relieving_factors
  12. ask_associated_symptoms
  13. ask_medications
  14. ask_allergies
  15. ask_exertional_component

HPI:
{'aggravating_factors': ['exercise brings it on'],
 'associated_symptoms': ['dizziness during the episode'],
 'character': 'Racing and pounding',
 'duration': 'about 5 minutes',
 'onset': 'This afternoon during my run',
 'relieving_factors': ['stopping exercise helps'],
 'severity': '7/10',
 'timing': 'Intermittent, comes on with exercise'}

POLICY ANSWERS (non-None):
{'aura_features': [],
 'chest_pain': False,
 'diaphoresis': True,
 'neurologic_symptom_terms': [],
 'radiation': [],
 'shortness_of_breath': False,
 'syncope_or_presyncope': True}

FLAGS: []


## Summary

In [9]:
total = len(all_results)
passed_count = sum(all_results)
failed = total - passed_count
print(f'\n==============================')
print(f'  Results: {passed_count}/{total} passed')
if failed:
    print(f'  FAILED:  {failed}')
else:
    print(f'  All checks passed!')
print(f'==============================')


  Results: 34/34 passed
  All checks passed!
